In [1]:
import pandas as pd
import kagglehub
import os
import shutil
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import time
import warnings
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
print("--- Thư viện đã sẵn sàng ---")
# Tải dataset
path = kagglehub.dataset_download("wardabilal/spotify-global-music-dataset-20092025")
print(f"Dataset đã tải về tại: {path}")

# Di chuyển vào thư mục dự án data/raw
raw_folder = "../data/raw/"
os.makedirs(raw_folder, exist_ok=True)

for file in os.listdir(path):
    if file.endswith(".csv"):
        shutil.copy(os.path.join(path, file), raw_folder)
        print(f"Đã sao chép: {file}")
# Đọc dữ liệu
df_2025 = pd.read_csv('../data/raw/spotify_data clean.csv')
df_history = pd.read_csv('../data/raw/track_data_final.csv')

# Gộp thô
df_merged = pd.concat([df_2025, df_history], ignore_index=True)

# Lọc các track_id duy nhất để tối ưu hóa việc gọi API
unique_tracks = df_merged.drop_duplicates(subset=['track_id'])
print(f"Số lượng bản ghi sau khi gộp: {len(df_merged)}")
print(f"Số lượng ID duy nhất cần truy vấn API: {len(unique_tracks)}")

c:\Users\ASUS\Downloads\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Thư viện đã sẵn sàng ---
Dataset đã tải về tại: C:\Users\ASUS\.cache\kagglehub\datasets\wardabilal\spotify-global-music-dataset-20092025\versions\1
Đã sao chép: spotify_data clean.csv
Đã sao chép: track_data_final.csv
Số lượng bản ghi sau khi gộp: 17360
Số lượng ID duy nhất cần truy vấn API: 8778


In [2]:
# Tải các biến môi trường từ file .env
load_dotenv()

# Khởi tạo xác thực Spotify
client_credentials_manager = SpotifyClientCredentials()
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

print("--- Bắt đầu gọi API Spotify lấy Audio Features ---")

# Chuyển cột track_id thành một list, loại bỏ các giá trị NaN nếu có
track_ids = unique_tracks['track_id'].dropna().tolist()

audio_features_list = []
batch_size = 100 # Giới hạn của Spotify API cho endpoint này là 100 tracks/request

# Vòng lặp chia nhỏ batch
for i in range(0, len(track_ids), batch_size):
    batch = track_ids[i:i + batch_size]
    try:
        # Gọi API lấy audio features cho batch hiện tại
        features = sp.audio_features(batch)
        
        # Spotify có thể trả về None cho một số track không có dữ liệu âm học
        # Cần lọc bỏ các giá trị None này
        features = [f for f in features if f is not None]
        
        audio_features_list.extend(features)
        
        # Nghỉ 0.5 - 1 giây để tránh bị đánh dấu là spam request (Rate Limiting)
        time.sleep(0.5) 
        
        if i % 1000 == 0 and i > 0:
            print(f"Đã xử lý {i} / {len(track_ids)} tracks...")
            
    except Exception as e:
        print(f"Lỗi khi lấy dữ liệu ở batch {i}: {e}")

print(f"Đã lấy thành công đặc trưng âm học cho {len(audio_features_list)} bài hát.")

# Chuyển đổi list kết quả thành DataFrame
df_audio_features = pd.DataFrame(audio_features_list)

# Bước 3: Gộp (Merge) dữ liệu âm học vào dữ liệu gốc
# API Spotify trả về ID của track dưới cột tên là 'id'
# Đổi tên cột 'id' thành 'track_id' để dễ merge với dữ liệu ban đầu
if not df_audio_features.empty:
    df_audio_features = df_audio_features.rename(columns={'id': 'track_id'})
    
    # Loại bỏ một số cột không cần thiết mà API trả về (nếu muốn dataset gọn hơn)
    columns_to_drop = ['type', 'uri', 'track_href', 'analysis_url']
    df_audio_features = df_audio_features.drop(columns=columns_to_drop, errors='ignore')

    # Merge với df_merged ban đầu (LEFT JOIN)
    final_df = pd.merge(df_merged, df_audio_features, on='track_id', how='left')
    
    print(f"Số lượng cột sau khi merge: {len(final_df.columns)}")
    print("Mẫu dữ liệu sau khi merge:")
    print(final_df.head())
    
    # Lưu ra file CSV mới
    output_path = '../data/raw/spotify_data_with_features.csv'
    final_df.to_csv(output_path, index=False)
    print(f"Đã lưu file dữ liệu hoàn chỉnh tại: {output_path}")
else:
    print("Không lấy được dữ liệu audio features nào.")

--- Bắt đầu gọi API Spotify lấy Audio Features ---


HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=3EJS5LyekDim1Tf5rBFmZl,1oQW6G2ZiwMuHqlPpP27DB,7mdkjzoIYlf1rx9EtBpGmU,67rW0Zl7oB3qEpD5YWWE5w,15xptTfRBrjsppW0INUZjf,4ccpCcZYseq8VrPMK1EDs0,3QoQ3HqXTAjgEl9LbNMbYp,1YEZbdT417SfolPQzaoHs2,4pZ949nFW5SurwzE0TSe7I,0L0LgwFZ7UtBnRNQvSBty6,5mfAZwEr4jAVDnSY0HHuyH,4V1Sh9Y2z6e8HpzNOet5rC,0NZKfcua68wuZePhjp8N2I,6zFWxD9hkOPAIlgWAGyQGQ,66F6YyfnWy6cNc6bd3zl2C,22c6Yop5YVEAhw8UG6O1W1,0SoYN9bRGPj7qV6loQKCTO,19A7srqQSXoW14muGyiHjg,0XQQpq855N5rnUsFBXo2ml,0nBPI5KxKOeOtr24sdGpE6,2aMZEJZwWouRzOw2aSfKV9,2zzjpwG8wu3skr08m7rM6e,2MbsC60IxeyJNVRRgeqH4w,2Zq3Trov36G0TtrUt4ZklL,7rdHxPQimXkOcmUmsfGhxY,2H6cMXx0Kmj6XlbMx4BYi9,6DIrsJFPypT0Z3emDKnXYd,05aYqGC8NNxr4jiHbQXLG3,0C42YiaJQyb0OExJ89bRbs,1yLkASt2rIp9W5t3NuZrwQ,2zLnwikuCxJwMkEI8tqDN4,0kWQFzHzEPtvrKnwngvWyb,5NWYu2pYfLGu3iQB08RC4m,4cVw1IeHrFuVS58Glolglb,31lp34zEvY603Xjkz2hrOo,1nNgDKWWX9MU9cp5FrHXRN,4ljqKsDyZzicNpBIMvDaZ5,5hw7dvgBRLvlCiALWwujYz,6OTDBDVaK18xlne9rlGQok,1NXbNEAcPvY5G1xvfN57aA,6gRNWnK8b8B

Lỗi khi lấy dữ liệu ở batch 0: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=3EJS5LyekDim1Tf5rBFmZl,1oQW6G2ZiwMuHqlPpP27DB,7mdkjzoIYlf1rx9EtBpGmU,67rW0Zl7oB3qEpD5YWWE5w,15xptTfRBrjsppW0INUZjf,4ccpCcZYseq8VrPMK1EDs0,3QoQ3HqXTAjgEl9LbNMbYp,1YEZbdT417SfolPQzaoHs2,4pZ949nFW5SurwzE0TSe7I,0L0LgwFZ7UtBnRNQvSBty6,5mfAZwEr4jAVDnSY0HHuyH,4V1Sh9Y2z6e8HpzNOet5rC,0NZKfcua68wuZePhjp8N2I,6zFWxD9hkOPAIlgWAGyQGQ,66F6YyfnWy6cNc6bd3zl2C,22c6Yop5YVEAhw8UG6O1W1,0SoYN9bRGPj7qV6loQKCTO,19A7srqQSXoW14muGyiHjg,0XQQpq855N5rnUsFBXo2ml,0nBPI5KxKOeOtr24sdGpE6,2aMZEJZwWouRzOw2aSfKV9,2zzjpwG8wu3skr08m7rM6e,2MbsC60IxeyJNVRRgeqH4w,2Zq3Trov36G0TtrUt4ZklL,7rdHxPQimXkOcmUmsfGhxY,2H6cMXx0Kmj6XlbMx4BYi9,6DIrsJFPypT0Z3emDKnXYd,05aYqGC8NNxr4jiHbQXLG3,0C42YiaJQyb0OExJ89bRbs,1yLkASt2rIp9W5t3NuZrwQ,2zLnwikuCxJwMkEI8tqDN4,0kWQFzHzEPtvrKnwngvWyb,5NWYu2pYfLGu3iQB08RC4m,4cVw1IeHrFuVS58Glolglb,31lp34zEvY603Xjkz2hrOo,1nNgDKWWX9MU9cp5FrHXRN,4ljqKsDyZzicNpBIMvDaZ5,5hw7dvgBRLvlCiALWwujYz,6OTDBDVaK18xlne9rlG

HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=7CHzHT86zvEkeZpVPr6IgL,1eLfsiFKFVpvPXi1IFSZGO,2lCeVXvAGYUud5smJ9PhGv,7ksHySXQO1EMxmecq9k0SF,5jAKEKeNzRbcuKlWGv9cT1,7woJR9AI4OwLIpzzP6Y3Yy,4HUfvzEUU54VoSq5SWdOcJ,5PXUWF5p5Prmul75Rye7mb,6vqyk3mbDBv3npTpctYoka,7DmGjrkQml1VCIgvrdNW3G,4G7bbgD2DHORU4frrWoCXp,0TQ5bJ44VHnFZyC16folF0,5YZvDeQOUhItx80C7KJCfX,4zzfmeA6MXiM9O4anGZkq5,388JceJm7784qgHipfPKmr,3orYgsOb8x1HPVwzeZBCur,4Xj6u5UdqINhGItVdwCLZp,4HBeFgbY9VV7UuMltmCIw0,56sExwBBmnDoRErXqghqrW,77CvPAHOsHWl43rUV3zsg6,1CjFD8mc8f40JtIIK0NRGA,1mEQbSq0PCNjyL6oMgHKFQ,4hJNJqTcFptFRWfTjRpr3d,3NFs3XUduzBfvc5Bx1gmzh,51opBUruRqfn4bgIuf5nwU,5pobX21I8uZdHM8YOWItU7,6A331YYR5zoG6XixuHX7yE,4oT5126hWjtMCeZMu2RnSx,7ffAlbOWcI5u6NN6XHPfSS,65vAx13K5AlJ5IEpw2ZcTK,4iKIF8qI3dq6fntHeIdcAw,4XMtEsNnR4nOFeyTBQqxnp,4vQcqtphmJBR5R44F5iNvl,6AWldBTJJWOv0VbrNZXFEw,6iAWfzQSSHemnvL5yq1q2k,0RxdWAztwE2qKbtYk7NNWr,5amdAZWVhOqFHodIyf53bI,0MjkLEOp7lHilik4t8G9oJ,2Xunjp5lfCjPJKuiWEVA5O,0XXxTyNb3Q0DPxpf0kQi8P,4vcDr56fq4J

Lỗi khi lấy dữ liệu ở batch 200: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=7CHzHT86zvEkeZpVPr6IgL,1eLfsiFKFVpvPXi1IFSZGO,2lCeVXvAGYUud5smJ9PhGv,7ksHySXQO1EMxmecq9k0SF,5jAKEKeNzRbcuKlWGv9cT1,7woJR9AI4OwLIpzzP6Y3Yy,4HUfvzEUU54VoSq5SWdOcJ,5PXUWF5p5Prmul75Rye7mb,6vqyk3mbDBv3npTpctYoka,7DmGjrkQml1VCIgvrdNW3G,4G7bbgD2DHORU4frrWoCXp,0TQ5bJ44VHnFZyC16folF0,5YZvDeQOUhItx80C7KJCfX,4zzfmeA6MXiM9O4anGZkq5,388JceJm7784qgHipfPKmr,3orYgsOb8x1HPVwzeZBCur,4Xj6u5UdqINhGItVdwCLZp,4HBeFgbY9VV7UuMltmCIw0,56sExwBBmnDoRErXqghqrW,77CvPAHOsHWl43rUV3zsg6,1CjFD8mc8f40JtIIK0NRGA,1mEQbSq0PCNjyL6oMgHKFQ,4hJNJqTcFptFRWfTjRpr3d,3NFs3XUduzBfvc5Bx1gmzh,51opBUruRqfn4bgIuf5nwU,5pobX21I8uZdHM8YOWItU7,6A331YYR5zoG6XixuHX7yE,4oT5126hWjtMCeZMu2RnSx,7ffAlbOWcI5u6NN6XHPfSS,65vAx13K5AlJ5IEpw2ZcTK,4iKIF8qI3dq6fntHeIdcAw,4XMtEsNnR4nOFeyTBQqxnp,4vQcqtphmJBR5R44F5iNvl,6AWldBTJJWOv0VbrNZXFEw,6iAWfzQSSHemnvL5yq1q2k,0RxdWAztwE2qKbtYk7NNWr,5amdAZWVhOqFHodIyf53bI,0MjkLEOp7lHilik4t8G9oJ,2Xunjp5lfCjPJKuiW

...

Lỗi khi lấy dữ liệu ở batch 8100: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=1Je1IMUlBXcx1Fz0WE7oPT,1DIXPcTDzTj8ZMHt3PDt8p,6urCAbunOQI4bLhmGpX7iS,0snDRsjNRMJhm7SVja8l6J,3n2ujlr3mCscnFFpepXAIy,1qPbGZqppFwLwcBC1JQ6Vr,5wj4E6IsrVtn8IBJQOd0Cl,7ppPZa3TRUSGKaks9wH7VT,6iy4PoAuZBMvtrlDX4VxC7,3IvcWWehbBcBR4YZxlGM9R,6SI4JD7iyQ0rrudhCNtMv0,6GtX0jaNL8IjVQfrDBx81z,7ygpwy2qP3NbrxVkHvUhXY,2u2udGmop1z67EPpr91km7,33ZXjLCpiINn8eQIDYEPTD,7N1Vjtzr1lmmCW9iasQ8YO,50I6fV2otuiDmigTtN6WMY,73CKjW3vsUXRpy3NnX4H7F,6UO72VSXEONxdfLyABihs9,7MGFhIe9CBI5baTwLMj7iE,3Yvi5NkUrSppVwrMHYkB6u,1nWLqfovfb9QcYM1irpc6V,56sWRGGeRFWgFhrMOgOyZC,2302lUwfZ4S4dVyPOCDFnQ,36YJWsgSCEMFsv9Y2jxI4U,48mJX8glOrQkrSdVBjc0Wb,22yxHt6UqZpH7tP6W4PooI,0XgpiStoxq1IJncYlPrvZ5,0xHch89AN9RlN8Nm4kJzs3,551qy5vUgrUfEUc4dCNfht,48YkpE5enOln5c7jXSTuHL,725QAOexNLmGZiFMN7U8pF,7erXAfnatmoUow8DnbseOY,4LrGrY6IyJUlkFu6Ms4myU,4tReFKumS5bcFahdXDiM1b,57R1nBluakXaayH9EzgwZa,7pm8e1QG0XJDZ2nJZTeDBw,1Hv1VTm8zeOeybub15mA2R,38QnxZJMktnt96bx

HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=2OErSh4oVVAUll0vHxdr6p,6sawhOATOEe0cGzXJfMJQE,5Hyr47BBGpvOfcykSCcaw9,2Di0qFNb7ATroCGB3q0Ka7,1FPSkRkDlthbAceSIIWc4C,648BMGrt98kUbLo24A4vgj,7G7tgVYORlDuVprcYHuFJh,4ByEFOBuLXpCqvO1kw8Wdm,1LwP9g1Hjbs64jXM2Qsxry,396Oe9cFXg8EP8SO0hl4gG,1PtQJZVZIdWIYdARpZRDFO,5yGTQzYbEdY6B9RFZJypgt,75FEaRjZTKLhTrFGsfMUXR,2WfaOiMkCvy7F5fcp2zZ8L,4RvWPyQ5RL0ao9LPZeSouE,6w6I3AFRv7tQMmUTgAghUB,4XUvVbcITJLPEmA59suKV3,7yzbimr8WVyAtBX3Eg6UL9,5vmRQ3zELMLUQPo2FLQ76x,6BrMEbPSSj55nQhkgf6DnE,0LATteOxdYq4vXTNZ97r77,7FwBtcecmlpc1sLySPXeGE,0X1sqQ652p1sceKM2nJlIJ,1MqGKtY9L5qjPi8s7gX645,0pUVeEgZuNyFzIMKp67RbS,4y1LsJpmMti1PfRQV9AWWe,28WmNsclKsrVmdv3tDmoYU,1JSTJqkT5qHq8MDJnJbRE1,1TfqLAPs4K3s2rJMoCokcS,7J1uxwnxfQLu4APicE5Rnj,2374M0fQpWi3dLnB54qaLX,4TnUKixNWMfajncgdSwFoi,4GDmAT5ZZyHdBo32UYDIvM,2Cdvbe2G4hZsnhNMKyGrie,0ehVk4Py6iyLpfruxVYq4S,5RKQ5NdjSh2QzD4MaunT91,0cGG2EouYCEEC3xfa0tDFV,6EPRKhUOdiFSQwGBRBbvsZ,3K7Q9PHUWPTaknlbFPThn2,08mG3Y1vljYA6bvDt4Wqkj,4w3tQBXhn53

Lỗi khi lấy dữ liệu ở batch 8400: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=2OErSh4oVVAUll0vHxdr6p,6sawhOATOEe0cGzXJfMJQE,5Hyr47BBGpvOfcykSCcaw9,2Di0qFNb7ATroCGB3q0Ka7,1FPSkRkDlthbAceSIIWc4C,648BMGrt98kUbLo24A4vgj,7G7tgVYORlDuVprcYHuFJh,4ByEFOBuLXpCqvO1kw8Wdm,1LwP9g1Hjbs64jXM2Qsxry,396Oe9cFXg8EP8SO0hl4gG,1PtQJZVZIdWIYdARpZRDFO,5yGTQzYbEdY6B9RFZJypgt,75FEaRjZTKLhTrFGsfMUXR,2WfaOiMkCvy7F5fcp2zZ8L,4RvWPyQ5RL0ao9LPZeSouE,6w6I3AFRv7tQMmUTgAghUB,4XUvVbcITJLPEmA59suKV3,7yzbimr8WVyAtBX3Eg6UL9,5vmRQ3zELMLUQPo2FLQ76x,6BrMEbPSSj55nQhkgf6DnE,0LATteOxdYq4vXTNZ97r77,7FwBtcecmlpc1sLySPXeGE,0X1sqQ652p1sceKM2nJlIJ,1MqGKtY9L5qjPi8s7gX645,0pUVeEgZuNyFzIMKp67RbS,4y1LsJpmMti1PfRQV9AWWe,28WmNsclKsrVmdv3tDmoYU,1JSTJqkT5qHq8MDJnJbRE1,1TfqLAPs4K3s2rJMoCokcS,7J1uxwnxfQLu4APicE5Rnj,2374M0fQpWi3dLnB54qaLX,4TnUKixNWMfajncgdSwFoi,4GDmAT5ZZyHdBo32UYDIvM,2Cdvbe2G4hZsnhNMKyGrie,0ehVk4Py6iyLpfruxVYq4S,5RKQ5NdjSh2QzD4MaunT91,0cGG2EouYCEEC3xfa0tDFV,6EPRKhUOdiFSQwGBRBbvsZ,3K7Q9PHUWPTaknlb

HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=3PfIrDoz19wz7qK7tYeu62,3zkWCteF82vJwv0hRLba76,1UrwJzlNC2oaTlxj1OZmcu,5TbzAWWc5eJaANpA9kfGCd,1dI77VhaLcQSgQLSnIs03D,6D60klaHqbCl9ySc8VcRss,2ThsBNtjuUEERqcpXQEoYR,71vsEyBd4X1D5BUmLdFSVH,1mXVgsBdtIVeCLJnSnmtdV,3bidbhpOYeV4knp8AIu8Xn,1vTntttEXlVHk4uj1vtVEe,1Yk0cQdMLx5RzzFTYwmuld,1MtUq6Wp1eQ8PC6BbPCj8P,4B0JvthVoAAuygILe3n4Bs,4wCmqSrbyCgxEXROQE6vtV,5PKWUDfQFtc5qqo8cs1gQp,3DXncPQOG4VBw3QHh3S817,5Sf3GyLEAzJXxZ5mbCPXTu,4cluDES4hQEUhmXj6TXkSo,2fQxE0jVrjNMT9oJAXtSJR,1YiD079ne2nl9TEOrw43Gh,1qwno7xb5mJe71xtMS6jl2,5Eh1nj7IjV9lwpcKAkidyY,2262bWmqomIaJXwCRHr13j,6O5TrlFWTYvznd9fMC0VvU,3H37ijjdmKz0dQLgUGzpjx,6DEMMeWXfmFAXgDUMMzeg6,7wPTyxE1PNemZuyuOOaQ8q,62oaDoBveZxvZb94ziiMXi,1brwdYwjltrJo7WHpIvbYt,2tpfxAXiI52znho4WE3XFA,7s5VQqrjBtrBgZL4pEa46S,1kqc6U8hVYZhY0gFGQclCz,6VwBbL8CzPiC4QV66ay7oR,53iuhJlwXhSER5J2IYYv1W,3Dknan6iov4lBCwyHHinhS,0tKcYR2II1VCQWT79i5NrW,4qKcDkK6siZ7Jp1Jb4m0aL,4ZMJshw6UOvaTWWThYtEoW,0eOirxo4iDsRMxNuOVeqnO,7fLzbEOBOae

Lỗi khi lấy dữ liệu ở batch 8700: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=3PfIrDoz19wz7qK7tYeu62,3zkWCteF82vJwv0hRLba76,1UrwJzlNC2oaTlxj1OZmcu,5TbzAWWc5eJaANpA9kfGCd,1dI77VhaLcQSgQLSnIs03D,6D60klaHqbCl9ySc8VcRss,2ThsBNtjuUEERqcpXQEoYR,71vsEyBd4X1D5BUmLdFSVH,1mXVgsBdtIVeCLJnSnmtdV,3bidbhpOYeV4knp8AIu8Xn,1vTntttEXlVHk4uj1vtVEe,1Yk0cQdMLx5RzzFTYwmuld,1MtUq6Wp1eQ8PC6BbPCj8P,4B0JvthVoAAuygILe3n4Bs,4wCmqSrbyCgxEXROQE6vtV,5PKWUDfQFtc5qqo8cs1gQp,3DXncPQOG4VBw3QHh3S817,5Sf3GyLEAzJXxZ5mbCPXTu,4cluDES4hQEUhmXj6TXkSo,2fQxE0jVrjNMT9oJAXtSJR,1YiD079ne2nl9TEOrw43Gh,1qwno7xb5mJe71xtMS6jl2,5Eh1nj7IjV9lwpcKAkidyY,2262bWmqomIaJXwCRHr13j,6O5TrlFWTYvznd9fMC0VvU,3H37ijjdmKz0dQLgUGzpjx,6DEMMeWXfmFAXgDUMMzeg6,7wPTyxE1PNemZuyuOOaQ8q,62oaDoBveZxvZb94ziiMXi,1brwdYwjltrJo7WHpIvbYt,2tpfxAXiI52znho4WE3XFA,7s5VQqrjBtrBgZL4pEa46S,1kqc6U8hVYZhY0gFGQclCz,6VwBbL8CzPiC4QV66ay7oR,53iuhJlwXhSER5J2IYYv1W,3Dknan6iov4lBCwyHHinhS,0tKcYR2II1VCQWT79i5NrW,4qKcDkK6siZ7Jp1Jb4m0aL,4ZMJshw6UOvaTWWT